# Phase 3 — Clean Transaction Layer

Explores `clean_insider_transactions`: normalized dates/numbers, taxonomies, quality flags, and the MVP alpha universe.

Build the table first:

```bash
python -m src.cleaning.build_clean_insider_transactions --reset
```

For a quick sample: add `--limit 100000`.

## 1. What `clean_insider_transactions` is

One row = one non-derivative transaction line joined to submission metadata and **one** reporting-owner row.
Filings with multiple insiders duplicate transaction rows (by design).

Primary key: `transaction_id` = SHA256(accession + nonderiv_sk + insider_cik).

## 2. Raw vs clean vs MVP alpha universe

| Layer | Table | Mutability |
|-------|-------|------------|
| Raw | `raw_*` | Immutable SEC text |
| Clean | `clean_insider_transactions` | Parsed + enriched; all form types kept |
| MVP | `include_mvp_alpha = 1` | Form 4, P/S, common equity, valid event, value ≥ threshold |

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.db import get_connection, get_database_path

EXPORT_DIR = PROJECT_ROOT / "data" / "exports" / "cleaning"
DB_PATH = get_database_path()
conn = get_connection(DB_PATH)

print(f"Project root: {PROJECT_ROOT}")
print(f"Database:     {DB_PATH}")
print(f"Exports:      {EXPORT_DIR}")

## 3. Row counts

In [ ]:
summary_path = EXPORT_DIR / "clean_insider_transactions_summary.csv"
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    counts = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS total_clean_rows,
            SUM(include_mvp_alpha) AS mvp_alpha_rows,
            SUM(is_valid_clean_event) AS valid_clean_event_rows
        FROM clean_insider_transactions
        """,
        conn,
    )
    display(counts)

## 4. Form type breakdown

In [ ]:
form_df = pd.read_sql_query(
    """
    SELECT document_type,
           SUM(is_form_3) AS form_3,
           SUM(is_form_4) AS form_4,
           SUM(is_form_5) AS form_5,
           SUM(is_amendment) AS amendments,
           COUNT(*) AS rows
    FROM clean_insider_transactions
    GROUP BY document_type
    ORDER BY rows DESC
    LIMIT 25
    """,
    conn,
)
display(form_df)

## 5. Transaction code breakdown

In [ ]:
code_df = pd.read_sql_query(
    """
    SELECT transaction_code, transaction_category, signal_direction,
           SUM(include_mvp_alpha) AS mvp_rows,
           COUNT(*) AS rows
    FROM clean_insider_transactions
    GROUP BY transaction_code, transaction_category, signal_direction
    ORDER BY rows DESC
    LIMIT 30
    """,
    conn,
)
display(code_df)

## 6. Security type breakdown

In [ ]:
sec_df = pd.read_sql_query(
    """
    SELECT normalized_security_type,
           SUM(security_include_clean_mvp) AS mvp_eligible_security,
           COUNT(*) AS rows
    FROM clean_insider_transactions
    GROUP BY normalized_security_type
    ORDER BY rows DESC
    """,
    conn,
)
display(sec_df)

## 7. Role breakdown

In [ ]:
role_df = pd.read_sql_query(
    """
    SELECT normalized_role, role_group,
           AVG(role_weight) AS avg_weight,
           COUNT(*) AS rows
    FROM clean_insider_transactions
    GROUP BY normalized_role, role_group
    ORDER BY rows DESC
    LIMIT 25
    """,
    conn,
)
display(role_df)

## 8. Quality flags

In [ ]:
flags_path = EXPORT_DIR / "clean_quality_flags_summary.csv"
if flags_path.exists():
    display(pd.read_csv(flags_path))
else:
    flags = [
        "is_missing_price", "is_zero_price", "is_negative_or_zero_shares",
        "is_filing_before_transaction", "is_tiny_transaction",
        "is_extreme_transaction_value", "is_valid_clean_event",
    ]
    rows = []
    for flag in flags:
        n = conn.execute(
            f"SELECT SUM({flag}) FROM clean_insider_transactions"
        ).fetchone()[0]
        rows.append({"quality_flag": flag, "row_count": n or 0})
    display(pd.DataFrame(rows))

## 9. Ticker cleaning and invalid placeholders

In [ ]:
ticker_qa = pd.read_sql_query(
    """
    SELECT is_valid_ticker, is_invalid_ticker_placeholder, is_multisymbol_ticker,
           COUNT(*) AS rows, SUM(include_mvp_alpha) AS mvp_rows
    FROM clean_insider_transactions
    GROUP BY 1, 2, 3 ORDER BY rows DESC
    """,
    conn,
)
display(ticker_qa)

## 10. Filing quality buckets

In [ ]:
lag_buckets = pd.read_sql_query(
    """
    SELECT filing_lag_bucket, COUNT(*) AS rows,
           SUM(include_mvp_alpha) AS mvp_rows, SUM(is_stale_filing) AS stale_rows
    FROM clean_insider_transactions
    GROUP BY filing_lag_bucket ORDER BY rows DESC
    """,
    conn,
)
display(lag_buckets)

## 11. Historical transaction date vs source year

In [ ]:
pre_src = PROJECT_ROOT / "data" / "exports" / "cleaning" / "pre_source_transaction_date_sample.csv"
if pre_src.exists():
    display(pd.read_csv(pre_src).head(20))
else:
    display(pd.read_sql_query(
        """SELECT transaction_year, source_year, transaction_year_source_gap, COUNT(*) AS rows
           FROM clean_insider_transactions WHERE is_pre_source_transaction_date = 1
           GROUP BY 1,2,3 ORDER BY rows DESC LIMIT 20""", conn))

## 12. MVP alpha universe preview

In [ ]:
preview_path = EXPORT_DIR / "mvp_alpha_universe_preview.csv"
if preview_path.exists():
    preview = pd.read_csv(preview_path)
else:
    preview = pd.read_sql_query(
        """
        SELECT ticker, issuer_name, insider_name, transaction_date,
               transaction_code, transaction_value, normalized_role,
               normalized_security_type, filing_lag_days
        FROM clean_insider_transactions
        WHERE include_mvp_alpha = 1
        ORDER BY transaction_value DESC
        LIMIT 1000
        """,
        conn,
    )
display(preview.head(20))
print(f"Preview rows: {len(preview):,}")

## 13. Recommended next step

Phase 4 should join market prices and compute forward returns on rows where `include_mvp_alpha = 1`, then layer insider role weights and scoring. No market data or alpha is calculated in Phase 3.